# gaussian-fit.ipynb
Fits a Gaussian to each g_z(O) peak above 1.2 and marks the peak positions and the peak-to-peak distance.

- **Runs on:** PC
- **Inputs:** `<d>.out` in `GZ_DATA_DIR`
- **Outputs:** `gZ/results_pc/gaussian-fit/data_peak_gZ_<d>.png`
- Run **Setup** first.

## Setup
Paths and `load_gz(d)`. **Change:** `GZ_DATA_DIR`.

In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path.cwd() / "results_pc"                      # outputs (runs on PC)
GZ_DATA_DIR = Path("/gibbs/arghavan/TODO_gZ_data")    # two-plate <d>.out files (d = distance x 10); set after PC listing
N_VOXELS = 170 * 170                                 # y-z voxels per x slab


def load_gz(d):
    """Two-plate g_z(O) for distance d (x10, e.g. 92): reads <d>.out, divides counts by N_VOXELS."""
    data = pd.read_csv(GZ_DATA_DIR / f"{d}.out", sep=r"\s+", header=None)
    return data[0], data[1] / N_VOXELS

## Gaussian fit, one distance (9.2 Å)
**Change:** distance range in `range(92, 93, 2)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import matplotlib.ticker as ticker 



for d in range(92, 93, 2):
    d_in_angstrom = d / 10
    x, number_density = load_gz(d)

    def gaussian(x, a, x0, sigma):
        return a * np.exp(-((x - x0)**2) / (2 * sigma**2))
    ##----

    # find raw peaks
    peak_indices, _ = find_peaks(number_density, height=1.2) 
    x_peaks_init = x[peak_indices]

    fitted_peaks = []


    # data plotting
    plt.figure(figsize=(10, 6))
    plt.plot(x, number_density, 'bo-', markersize=4, label='Two Plate Interference')
    # plt.hlines(y=1.00, xmin=min(x), xmax=max(x), color='black', linestyle='--', linewidth=1, label='Bulk Water')
    plt.hlines(y=1.00, xmin=18, xmax=32, color='black', linestyle='--', linewidth=1, label='Bulk Water')
    
    # gaussian fit
    for x0_guess in x_peaks_init:
        
        # mask ±0.3 around initial peak guess
        mask = (x > x0_guess - 0.25) & (x < x0_guess + 0.25)
        x_fit = x[mask]
        y_fit = number_density[mask]
        
        if len(x_fit) < 5:
            continue

        # initial guess [a, x0, sigma]
        p0 = [np.max(y_fit), x0_guess, 0.2]

        try:
            popt, _ = curve_fit(gaussian, x_fit, y_fit, p0=p0)
            a_fit, x_peak, sigma_fit = popt
            y_peak = gaussian(x_peak, *popt)

            # saving peak info
            fitted_peaks.append((x_peak, y_peak))
            # fitting
            x_dense = np.linspace(x_fit.min(), x_fit.max(), 500)
            peak_position = 25 + (d_in_angstrom/2) - x_peak
            plt.plot(x_dense, gaussian(x_dense, *popt), 'tab:cyan', linewidth = 1)
            plt.plot(x_peak, y_peak+0.2, 'rx', label=f'Peak position ≈  {peak_position:.2f} Å')

        except RuntimeError:
            print(f"Fit failed for peak near x = {x0_guess:.2f}")

    # show the distance 
    if len(fitted_peaks) >= 2:
        fitted_peaks.sort()  # sort by x
        x1, y1 = fitted_peaks[0]
        x2, y2 = fitted_peaks[1]
        y_line = max(y1, y2)
        plt.hlines(y=y_line+0.2, xmin=x1, xmax=x2, color='gray', linestyle='--')
        plt.text((x1 + x2) / 2, y_line + 0.3, f"{x2 - x1:.2f} Å", ha='center', fontsize=10, color='gray')
        # plt.text(x1, y_line + 0.1, f"{y1:.2f}", ha='center', fontsize=10, color='gray')
        # plt.text(x2, y_line + 0.1, f"{y2:.2f}", ha='center', fontsize=10, color='gray')

    plt.text(20, 5.5, f"Plate Distance: {d_in_angstrom} Å", ha='center', fontsize=11, color='dimgrey',
             bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.3'))
    # plt.text(25, 5.63, f"Plate Distance: {d_in_angstrom} Å", ha='center', fontsize=11, color='black',
    #          bbox=dict(facecolor='white', edgecolor='k', boxstyle='round,pad=0.3'))
    plt.title(f'\nAverage Number Density')
    plt.xlabel('z (Å)')
    plt.ylabel(r'$g_z(O)$')
    plt.legend()
    plt.ylim(-0.2, 6)
    plt.xlim(17.5,32.5)
    plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.gca().xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
    output_path = OUT / "gaussian-fit"
    output_path.mkdir(parents=True, exist_ok=True)
    plt.savefig(f"{output_path}/data_peak_gZ_{d_in_angstrom}.png", dpi=300)